In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import itertools
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.signal import savgol_filter, butter, filtfilt
from scipy.stats import spearmanr, pearsonr
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize, ListedColormap, BoundaryNorm, TwoSlopeNorm, LinearSegmentedColormap

%load_ext rpy2.ipython

Load in and process data, plotting params

In [ ]:
sns.set_theme(style="ticks", context="talk", font="Arial")

plt.rcParams.update({
    # Font + text sizes
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],  # fallbacks

    # Line widths
    "axes.linewidth": 0.5,   
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 2,     
    "ytick.major.size": 2,

    "svg.fonttype": "none",   
    "pdf.fonttype": 42,       
    "ps.fonttype": 42,      
    "text.usetex": False,    
})

In [ ]:
def lowpass(x, fs, cutoff_hz=0.3, order=3):
    b, a = butter(order, cutoff_hz / (fs / 2.0), btype="low")
    return filtfilt(b, a, np.asarray(x, float))


In [ ]:
df = pd.read_csv(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\ResidualTrialDF_Reward.csv')
df['Session'] = np.where(df['Session'] == 'Reward3', 'Reward', 'Monster')
df["Session"] = pd.Categorical(df["Session"], categories=['Reward', 'Monster'], ordered=True)

successdf = df.groupby(['Animal', 'Session', 'Trial'], observed=True).first().reset_index()[['Animal', 'Session', 'Trial', 'Time to Reward', 'Time to Monster']]
successdf['Rewarded'] = np.where(successdf['Time to Reward'].notna(), 1, 0)
successdf = successdf.groupby(['Animal', 'Session'], observed=True).agg("mean").reset_index()
goodanimals  = successdf[(successdf['Session'] == 'Reward') & (successdf['Rewarded'] >= 0.8)]['Animal'].to_list()

df = df[df['Animal'].isin(goodanimals)]
df['Site'] = df['Region']
df['Region'] = df['Site'].str[:2]

df = (df.sort_values(["Trial","Animal","Site","Time to Shelter"],
                     kind="mergesort", na_position="last")
        .reset_index(drop=True))

df['Outcome'] = np.where(df['Time to Reward'].notna(), 'Rewarded', 'Unrewarded')
df['MonsterCharge'] = np.where(df['Time to Monster'].notna(), 'Charge', 'NoCharge')
df['Outcome_Monster'] = df['Outcome']+df['MonsterCharge']

df['v_reward_radial'] = df['v_reward_radial']*-1

df = df[~((df['Region']=='TS')&(df['Animal']=='Wanda'))].copy()
df = df[~((df['Site']=='TSR')&(df['Animal']=='Chester'))].copy()

In [ ]:
parts = []
for g_names, g, in df.groupby(['Outcome_Monster', 'Session', 'Animal', 'Trial', 'Site']):
    outcome, session = g_names[0], g_names[1]
    fs  = float(1.0 / g['dt'].iloc[0])
    win = int(round(fs * 1.5)); win += 1 - (win % 2)
    win = max(win, 3)

    if len(g) < win:
        continue

    g['filtered_x']   = savgol_filter(g['X'].to_numpy(), win, 1, mode='interp')
    g['filtered_y']   = savgol_filter(g['Y'].to_numpy(), win, 1, mode='interp')
    dy                = savgol_filter(g['X'].to_numpy(), win, 1, deriv=1, delta=1/fs, mode='interp')
    g['d_filtered_x'] = lowpass(dy, fs, cutoff_hz=0.3)
    g.loc[abs(g['d_filtered_x']) <2, 'd_filtered_x' ] = 0

    g['approach'] = np.nan
    g['approach'] = g['approach'].astype('object')
    g.loc[g['filtered_x'] < 30,  'approach'] = 'Distal Approach' #0 #Distal Approach
    g.loc[g['filtered_x'] >= 30, 'approach'] = 'Proximal Approach' #20 #Proximal Approach
    g.loc[g['filtered_x'] >= 43, 'approach'] = 'Past Spout' #60 #Go past Spout
    g.loc[g['d_filtered_x'] < 0, 'approach'] = 'Retreat' #40 # retreat overrides bands

    rewardix = g['Time to Reward'].abs().idxmin()
    monsterix = (g['Time to Monster'].abs()).idxmin()
    consumptionix = np.nanmin([g.loc[(g.index > rewardix) & (g['approach'] == 'Retreat')].index.min(), g.loc[(g.index > rewardix) & (g['approach'] == 'Past Spout')].index.min()])
    g.loc[rewardix:consumptionix, 'approach'] = 'Licking'


    g.loc[(g['filtered_x'] <= 0), 'approach'] = np.nan

    lab  = g['approach'].to_numpy()
    prev = np.concatenate([lab[:1], lab[:-1]])

    transitions = (prev == 'Retreat') & (lab != 'Retreat')
    g['approach_counter'] = np.cumsum(transitions.astype(int))

    parts.append(g)
    
df = pd.concat(parts, axis=0).reset_index(drop=True)

df = df[df['approach'].notna()]

parts = []
for gname, g in df.groupby(['approach_counter', 'Outcome_Monster', 'Session', 'Animal', 'Trial', 'Site']):
    rmin, rmax = np.nanmin(g['Time to Reward']), np.nanmax(g['Time to Reward'])
    mmin, mmax = np.nanmin(g['Time to Monster']), np.nanmax(g['Time to Monster'])
    if np.isnan(rmin):
        g['Reward'] = 'Unrewarded'
    elif ((rmin<0) & (rmax<0)):
        g['Reward'] = 'Unrewarded'
    elif ((rmin<0) & (rmax>0)):
        g['Reward'] = 'Rewarded'
    elif ((rmin>0) & (rmax>0)):
        g['Reward'] = 'PostRewarded'

    if np.isnan(mmin):
        g['Monster'] = 'NoCharge'
    elif ((mmin<0) & (mmax<0)):
        g['Monster'] = 'NoCharge'
    elif ((mmin<0) & (mmax>0)):
        g['Monster'] = 'MonsterCharge'
    elif ((mmin>0) & (mmax>0)):
        g['Monster'] = 'MonsterCharging'

    parts.append(g)
df = pd.concat(parts, axis=0).reset_index(drop=True)

In [ ]:
df = df.copy()
df = df[~df['approach'].isin(['Licking', 'Past Spout'])]
df['AppRet'] = np.where(df['approach'] == 'Retreat', 'Retreat', 'Approach')

gcols = ["Animal", "Session", "Trial", "Site", "AppRet"]

C = abs(df["Z"].min())
df["Z_shift"] = df["Z"] + df.groupby(gcols)["Z"].transform("first")

df["romb_cum_Z"] = (
    df.groupby(gcols, sort=False, group_keys=False)
      .apply(lambda g: (0.5*(g["Z"] + g["Z"].shift(1)) * g["dt"])
             .fillna(0.0).cumsum())
)

# z-score within group
df["romb_cum_Z_zscore"] = (
    df.groupby(gcols, sort=False)["romb_cum_Z"]
      .transform(lambda x: (x - x.mean()) / x.std(ddof=1))
)


Figure 6C,6D,6E,6F

In [ ]:
X_col     = "X"
Y_col     = "romb_cum_Z_zscore"
TRIAL_COL = "Trial"
TIME_COL  = "Time to Shelter"
x_min, x_max = 0, 40

savgol_window = 11
savgol_poly   = 3
poly_deg      = 3

# Shuffle
n_perm       = 1000
shuffle_seed = 123

groups = [
    (key, sdf)
    for key, sdf in df.groupby(["Region", "Session", "AppRet"], sort=False)
    if not sdf.empty
]
n = len(groups)

fig, axes = plt.subplots(
    nrows=n, ncols=2,
    figsize=(12, 4.0 + 3.0*(n-1)),
    sharex=True,
    constrained_layout=True
)
if n == 1:
    axes = [axes]

coef_rows = []

for ax_row, ((site, session, appret), sub_df) in zip(axes, groups):

    sub = sub_df.dropna(subset=[X_col, Y_col, TIME_COL]).copy()

    sub[X_col] = pd.to_numeric(sub[X_col], errors="coerce")
    sub = sub.dropna(subset=[X_col, Y_col, TIME_COL])

    sub = sub[(sub[X_col] >= x_min) & (sub[X_col] <= x_max)]

    if TRIAL_COL not in sub.columns:
        raise KeyError(f"Expected trial column '{TRIAL_COL}' in dataframe")

    # Sort ordered by Time to Shelter
    sub = sub.sort_values(["Animal", TRIAL_COL, TIME_COL])

    if not np.issubdtype(sub[X_col].dtype, np.integer):
        nbins = 50
        sub[X_col] = sub[X_col].clip(
            lower=x_min,
            upper=np.nextafter(x_max, -np.inf)
        )
        bins = np.linspace(x_min, x_max, nbins + 1)
        sub["X_bin"] = pd.cut(
            sub[X_col], bins=bins,
            labels=False, include_lowest=True, right=True
        )
        centers = (bins[:-1] + bins[1:]) / 2
        center_map = pd.Series(centers, index=pd.RangeIndex(nbins))
        sub["X_mid"] = sub["X_bin"].map(center_map)
        x_used = "X_mid"
    else:
        x_used = X_col

    # BUILD SHUFFLE (circular, within trial)
    label_col = "X_mid" if "X_mid" in sub.columns else x_used
    if label_col not in sub.columns:
        sub[label_col] = sub[x_used]

    rng0 = np.random.RandomState(shuffle_seed)
    perm_chunks = []

    for i in range(n_perm):
        rng = np.random.RandomState(rng0.randint(0, 2**31 - 1))

        tmp = sub[["Animal", TRIAL_COL, TIME_COL, label_col, Y_col]].copy()
        tmp["Y_shuf"] = np.nan

        # Circularly shift Y within each Animal × Trial
        for (ani, trial), g in tmp.groupby(["Animal", TRIAL_COL], sort=False):
            n_rows = len(g)
            vals = g[Y_col].values

            if n_rows > 1:
                # uniform random circular shift across the whole trial
                shift = rng.randint(0, n_rows)
                vals = np.roll(vals, shift)

            tmp.loc[g.index, "Y_shuf"] = vals

        # Collapse over trials
        m = (
            tmp.groupby(["Animal", label_col], observed=True)["Y_shuf"]
               .mean()
               .reset_index()
               .rename(columns={label_col: "X_used", "Y_shuf": "Y_perm"})
        )
        perm_chunks.append(m)

    sub_shuffle = (
        pd.concat(perm_chunks, ignore_index=True)
          .groupby(["Animal", "X_used"], observed=True)["Y_perm"]
          .mean()
          .reset_index()
          .rename(columns={"Y_perm": Y_col})
    )

    datasets = [
        ("original", sub[["Animal", x_used, Y_col]].rename(columns={x_used: "X_used"}), ax_row[0]),
        ("shuffle",  sub_shuffle[["Animal", "X_used", Y_col]], ax_row[1])
    ]

    for control_label, df_in, ax_here in datasets:
        sns.despine(fig, left=True, bottom=True)

        g = (
            df_in.groupby(["Animal", "X_used"], as_index=False, observed=True)[Y_col]
                .mean()
                .sort_values(["X_used", "Animal"])
        )
        wide = g.pivot(index="X_used", columns="Animal", values=Y_col).sort_index()

        if "palette" in globals() and isinstance(palette, dict):
            pal = {a: palette.get(a, "0.4") for a in wide.columns}
        else:
            pal = dict(zip(wide.columns, sns.color_palette(n_colors=max(1, wide.shape[1]))))

        for animal in wide.columns:
            y_ser = wide[animal].dropna()
            x_vals = y_ser.index.values
            y_vals = y_ser.values

            if len(y_vals) >= 3:
                win = savgol_window if (savgol_window % 2 == 1) else savgol_window + 1
                win = min(win, len(y_vals) if (len(y_vals) % 2 == 1) else len(y_vals) - 1)
            else:
                win = None

            y_plot = (
                savgol_filter(
                    y_vals,
                    window_length=win,
                    polyorder=min(savgol_poly, win-1),
                    mode="interp"
                )
                if (win and win >= 3 and win > savgol_poly) else y_vals
            )

            lw    = 1.8 if control_label == "original" else 1.3
            alpha = 0.9 if control_label == "original" else 0.6
            ax_here.plot(x_vals, y_plot, lw=lw, alpha=alpha, color=pal[animal])

            # polynomial fit per animal
            mask = np.isfinite(x_vals) & np.isfinite(y_plot)
            if mask.sum() > poly_deg:
                coefs = np.polyfit(x_vals[mask], y_plot[mask], deg=poly_deg)
                yhat  = np.poly1d(coefs)(x_vals[mask])
                ss_res = np.sum((y_plot[mask] - yhat)**2)
                ss_tot = np.sum((y_plot[mask] - y_plot[mask].mean())**2)
                r2     = np.nan if ss_tot == 0 else 1.0 - ss_res/ss_tot

                rec = {
                    "Region": site, "Session": session, "AppRet": appret,
                    "Animal": animal,
                    "trace": "smoothed" if (win is not None) else "raw",
                    "deg": poly_deg, "n_points": int(mask.sum()),
                    "r2": float(r2), "control": control_label,
                }
                for i, c in enumerate(coefs):
                    rec[f"a{poly_deg - i}"] = float(c)
                for k in range(poly_deg-1, -1, -1):
                    rec.setdefault(f"a{k}", 0.0)
                coef_rows.append(rec)

        mean_across = wide.mean(axis=1, skipna=True)
        sem_across  = wide.sem(axis=1, skipna=True)
        ci_lo = mean_across - 1.96 * sem_across
        ci_hi = mean_across + 1.96 * sem_across

        mean_across = mean_across.sort_index()
        ci_lo = ci_lo.loc[mean_across.index]
        ci_hi = ci_hi.loc[mean_across.index]

        ax_here.plot(mean_across.index, mean_across.values, lw=3, color="k", zorder=10)
        ax_here.fill_between(
            mean_across.index,
            ci_lo.values,
            ci_hi.values,
            color="k",
            alpha=0.12,
            linewidth=0,
            zorder=5
        )

        x = mean_across.index.values
        y = mean_across.values
        if len(x) > poly_deg and np.isfinite(y).sum() > poly_deg:
            coefs = np.polyfit(x, y, deg=poly_deg)
            poly  = np.poly1d(coefs)
            x_fit = np.linspace(x_min, x_max, 200)
            y_fit = poly(x_fit)
            ax_here.plot(x_fit, y_fit, 'r--', lw=2.2)

            eq = (
                f"y = {coefs[0]:.3g}x³ + {coefs[1]:.3g}x² + {coefs[2]:.3g}x + {coefs[3]:.3g}"
                if poly_deg == 3 else "poly fit"
            )
            ax_here.text(
                0.02, 0.95, eq,
                transform=ax_here.transAxes,
                fontsize=9, color='r', ha='left', va='top',
                bbox=dict(facecolor='white', alpha=0.6, edgecolor='none')
            )

            yhat  = poly(x)
            ss_res = np.sum((y - yhat)**2)
            ss_tot = np.sum((y - y.mean())**2)
            r2     = np.nan if ss_tot == 0 else 1.0 - ss_res/ss_tot

            rec = {
                "Region": site, "Session": session, "AppRet": appret,
                "Animal": "__ACROSS__", "trace": "across_mean",
                "deg": poly_deg, "n_points": int(np.isfinite(y).sum()),
                "r2": float(r2), "control": control_label,
            }
            for i, c in enumerate(coefs):
                rec[f"a{poly_deg - i}"] = float(c)
            for k in range(poly_deg-1, -1, -1):
                rec.setdefault(f"a{k}", 0.0)
            coef_rows.append(rec)

        ax_here.set_title(f"{site} | {session} | {appret} ({control_label})")
        ax_here.set_xlabel(X_col)
        ax_here.set_ylabel(Y_col)
        ax_here.set_xlim(x_min, x_max)

for ax_row in axes:
    ymins, ymaxs = zip(*(ax.get_ylim() for ax in ax_row))
    row_ymin = min(ymins)
    row_ymax = max(ymaxs)
    for ax in ax_row:
        ax.set_ylim(row_ymin, row_ymax)

plt.savefig(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\ValueMap_Lineplot_CircularShuffle.svg')
plt.show()

# coefs
coeffs_df = pd.DataFrame(coef_rows).copy()
ordered = [
    "Region","Session","AppRet","Animal","control","trace",
    "deg","n_points","r2","a3","a2","a1","a0"
]
for col in ordered:
    if col not in coeffs_df.columns:
        coeffs_df[col] = np.nan

coeffs_df = coeffs_df[
    [c for c in ordered if c in coeffs_df.columns] +
    [c for c in coeffs_df.columns if c not in ordered]
]

print(coeffs_df.head())
# coeffs_df.to_csv("poly_coefficients_original_vs_shuffle.csv", index=False)


In [ ]:
X_col     = "X"
Y_col     = "romb_cum_Z_zscore"
TRIAL_COL = "Trial"
TIME_COL  = "Time to Shelter"

x_min, x_max = 0, 40

nbins = 40

savgol_window = 11
savgol_poly   = 3

# Circular shuffle
n_perm       = 1000
shuffle_seed = 123

cmap = "coolwarm"
center_at_zero = True
vmin, vmax = -0.5, 0.5

savepath = r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\ValueMap_Heatmap_CircularShuffle.svg'

groups = [
    (key, sdf)
    for key, sdf in df.groupby(["Region", "Session", "AppRet"], sort=False)
    if not sdf.empty
]
n = len(groups)

fig, axes = plt.subplots(
    nrows=n, ncols=2,
    figsize=(12, 4.0 + 3.0*(n-1)),
    sharex=True,
    constrained_layout=True
)
if n == 1:
    axes = [axes]

sns.despine(fig=fig, left=True, bottom=True)

heatmaps_orig, heatmaps_shuf, titles = [], [], []

bins = np.linspace(x_min, x_max, nbins + 1)
centers = (bins[:-1] + bins[1:]) / 2 
center_map = pd.Series(centers, index=pd.RangeIndex(nbins))

for ((site, session, appret), sub_df) in groups:

    sub = sub_df.dropna(subset=[X_col, Y_col, TIME_COL, TRIAL_COL]).copy()
    sub[X_col] = pd.to_numeric(sub[X_col], errors="coerce")
    sub = sub.dropna(subset=[X_col, Y_col, TIME_COL])
    sub = sub[(sub[X_col] >= x_min) & (sub[X_col] <= x_max)]
    if sub.empty:
        continue

    sub = sub.sort_values(["Animal", TRIAL_COL, TIME_COL])
    sub[X_col] = sub[X_col].clip(lower=x_min, upper=np.nextafter(x_max, -np.inf))
    sub["X_bin"] = pd.cut(sub[X_col], bins=bins, labels=False, include_lowest=True, right=True)
    sub["X_used"] = sub["X_bin"].map(center_map)

    g = (
        sub.groupby(["Animal", "X_used"], as_index=False, observed=True)[Y_col]
           .mean()
           .sort_values(["X_used", "Animal"])
    )
    wide = g.pivot(index="X_used", columns="Animal", values=Y_col).sort_index()

    prof_orig = (
        wide.mean(axis=1, skipna=True)
            .reindex(centers)
            .interpolate(limit_direction="both")
    )

    vals_orig = prof_orig.values
    if len(vals_orig) >= 3:
        win = savgol_window if (savgol_window % 2 == 1) else savgol_window + 1
        win = min(win, len(vals_orig) if (len(vals_orig) % 2 == 1) else len(vals_orig) - 1)
        if win >= 3 and win > savgol_poly:
            vals_orig = savgol_filter(
                vals_orig,
                window_length=win,
                polyorder=min(savgol_poly, win - 1),
                mode="interp"
            )

    # circular shuffle
    rng0 = np.random.RandomState(shuffle_seed)
    sum_vec = np.zeros(len(centers), dtype=float)
    count_vec = np.zeros(len(centers), dtype=int)

    base = sub[["Animal", TRIAL_COL, TIME_COL, "X_used", Y_col]].copy()

    for _ in range(n_perm):
        rng = np.random.RandomState(rng0.randint(0, 2**31 - 1))
        tmp = base.copy()
        tmp["Y_shuf"] = np.nan

        # circular shift Y
        for (ani, trial), gtr in tmp.groupby(["Animal", TRIAL_COL], sort=False):
            vals = gtr[Y_col].to_numpy()
            n_rows = len(vals)
            if n_rows > 1:
                shift = rng.randint(0, n_rows)
                vals = np.roll(vals, shift)
            tmp.loc[gtr.index, "Y_shuf"] = vals

        g_perm = (
            tmp.groupby(["Animal", "X_used"], observed=True)["Y_shuf"]
               .mean()
               .reset_index()
        )
        wide_perm = g_perm.pivot(index="X_used", columns="Animal", values="Y_shuf").sort_index()

        prof = (
            wide_perm.mean(axis=1, skipna=True)
                    .reindex(centers)
                    .interpolate(limit_direction="both")
        )

        v = prof.values.astype(float)
        m = np.isfinite(v)
        sum_vec[m] += v[m]
        count_vec[m] += 1

    with np.errstate(invalid="ignore", divide="ignore"):
        vals_shuf = sum_vec / np.maximum(count_vec, 1)

    if len(vals_shuf) >= 3:
        win = savgol_window if (savgol_window % 2 == 1) else savgol_window + 1
        win = min(win, len(vals_shuf) if (len(vals_shuf) % 2 == 1) else len(vals_shuf) - 1)
        if win >= 3 and win > savgol_poly:
            vals_shuf = savgol_filter(
                vals_shuf,
                window_length=win,
                polyorder=min(savgol_poly, win - 1),
                mode="interp"
            )

    heatmaps_orig.append(vals_orig)
    heatmaps_shuf.append(vals_shuf)
    titles.append((site, session, appret))

all_vals = [np.asarray(v, dtype=float) for v in (heatmaps_orig + heatmaps_shuf)]
data_vmin = vmin if vmin is not None else np.nanmin([np.nanmin(v) for v in all_vals])
data_vmax = vmax if vmax is not None else np.nanmax([np.nanmax(v) for v in all_vals])

norm = TwoSlopeNorm(vcenter=0, vmin=data_vmin, vmax=data_vmax) if center_at_zero else None

y_edges = np.array([0, 1])

im = None
for ax_row, (vals_o, vals_s, (site, session, appret)) in zip(axes, zip(heatmaps_orig, heatmaps_shuf, titles)):

    arr_o = np.asarray(vals_o, dtype=float).reshape(1, -1)
    im = ax_row[0].pcolormesh(
        bins, y_edges, arr_o,
        cmap=cmap,
        norm=norm if center_at_zero else None,
        vmin=None if center_at_zero else data_vmin,
        vmax=None if center_at_zero else data_vmax,
        shading="flat"
    )
    ax_row[0].set_title(f"{site} | {session} | {appret} (original)")
    ax_row[0].set_xlabel(X_col)
    ax_row[0].set_ylabel(Y_col)
    ax_row[0].set_yticks([])
    ax_row[0].set_xlim(x_min, x_max)
    ax_row[0].set_ylim(0, 1)

    # shuffle
    arr_s = np.asarray(vals_s, dtype=float).reshape(1, -1)
    ax_row[1].pcolormesh(
        bins, y_edges, arr_s,
        cmap=cmap,
        norm=norm if center_at_zero else None,
        vmin=None if center_at_zero else data_vmin,
        vmax=None if center_at_zero else data_vmax,
        shading="flat"
    )
    ax_row[1].set_title(f"{site} | {session} | {appret} (shuffle)")
    ax_row[1].set_xlabel(X_col)
    ax_row[1].set_ylabel(Y_col)
    ax_row[1].set_yticks([])
    ax_row[1].set_xlim(x_min, x_max)
    ax_row[1].set_ylim(0, 1)

cbar = fig.colorbar(im, ax=axes, orientation="vertical", shrink=0.9, pad=0.02)
cbar.set_label(Y_col, rotation=270, labelpad=12)

plt.savefig(savepath)
plt.show()


Figure 6G,H

In [ ]:
X_col     = "X"
Y_col     = "romb_cum_Z_zscore"
TRIAL_COL = "Trial"
TIME_COL  = "Time to Shelter"
x_min, x_max = 0, 40

nbins = 40
n_perm = 1000
shuffle_seed = 123

groups = [
    (key, sdf)
    for key, sdf in df.groupby(["Region", "Session", "AppRet"], sort=False)
    if not sdf.empty
]

trial_rows_raw = []
trial_rows_shuf = []

rng0 = np.random.RandomState(shuffle_seed)

for (site, session, appret), sub_df in groups:
    sub = sub_df.dropna(subset=[X_col, Y_col, TIME_COL, TRIAL_COL, "Animal"]).copy()
    sub[X_col] = pd.to_numeric(sub[X_col], errors="coerce")
    sub = sub.dropna(subset=[X_col, Y_col, TIME_COL])

    sub = sub[(sub[X_col] >= x_min) & (sub[X_col] <= x_max)]
    if sub.empty:
        continue

    if TRIAL_COL not in sub.columns:
        raise KeyError(f"Expected trial column '{TRIAL_COL}' in dataframe")

    sub = sub.sort_values(["Animal", TRIAL_COL, TIME_COL])

    if not np.issubdtype(sub[X_col].dtype, np.integer):
        sub[X_col] = sub[X_col].clip(lower=x_min, upper=np.nextafter(x_max, -np.inf))
        bins = np.linspace(x_min, x_max, nbins + 1)
        sub["X_bin"] = pd.cut(sub[X_col], bins=bins,
                              labels=False, include_lowest=True, right=True)
        centers = (bins[:-1] + bins[1:]) / 2
        center_map = pd.Series(centers, index=pd.RangeIndex(nbins))
        sub["X_mid"] = sub["X_bin"].map(center_map)
        label_col = "X_mid"
    else:
        label_col = X_col

    raw_trial = (
        sub.groupby(["Animal", TRIAL_COL, label_col], observed=True)[Y_col]
           .mean()
           .reset_index()
           .rename(columns={label_col: "X_used", Y_col: "Y_mean"})
    )
    raw_trial["Region"]  = site
    raw_trial["Session"] = session
    raw_trial["AppRet"]  = appret
    raw_trial["control"] = "original"
    trial_rows_raw.append(raw_trial)

    base = sub[["Animal", TRIAL_COL, TIME_COL, label_col, Y_col]].copy()

    acc_sum = None 
    acc_n   = None 

    for _ in range(n_perm):
        rng = np.random.RandomState(rng0.randint(0, 2**31 - 1))

        tmp = base.copy()
        tmp["Y_shuf"] = np.nan

        # circular shift Y 
        for (ani, trial), g in tmp.groupby(["Animal", TRIAL_COL], sort=False):
            vals = g[Y_col].to_numpy()
            n_rows = len(vals)
            if n_rows > 1:
                shift = rng.randint(0, n_rows)
                vals = np.roll(vals, shift)
            tmp.loc[g.index, "Y_shuf"] = vals

        m = (
            tmp.groupby(["Animal", TRIAL_COL, label_col], observed=True)["Y_shuf"]
               .mean()
               .reset_index()
               .rename(columns={label_col: "X_used", "Y_shuf": "Y_perm"})
        )

        s = m.set_index(["Animal", TRIAL_COL, "X_used"])["Y_perm"]

        if acc_sum is None:
            acc_sum = s.copy()
            acc_n   = pd.Series(1.0, index=s.index)
        else:
            acc_sum = acc_sum.add(s, fill_value=0.0)
            acc_n   = acc_n.add(pd.Series(1.0, index=s.index), fill_value=0.0)

    shuf_mean = (acc_sum / acc_n).rename("Y_mean").reset_index()
    shuf_mean["Region"]  = site
    shuf_mean["Session"] = session
    shuf_mean["AppRet"]  = appret
    shuf_mean["control"] = "shuffle"
    shuf_mean["n_perm_eff"] = acc_n.values 

    trial_rows_shuf.append(shuf_mean)

raw_trial_df  = pd.concat(trial_rows_raw,  ignore_index=True) if trial_rows_raw  else pd.DataFrame()
shuf_trial_df = pd.concat(trial_rows_shuf, ignore_index=True) if trial_rows_shuf else pd.DataFrame()

lineplot_trial_df = pd.concat([raw_trial_df, shuf_trial_df], ignore_index=True)

lineplot_animal_df = (
    lineplot_trial_df
      .groupby(["Region","Session","AppRet","control","Animal","X_used"], observed=True, as_index=False)
      .agg(Y_mean=("Y_mean","mean"))
)


mean_lineplot_trial_df = lineplot_trial_df.groupby(['Animal','Region', 'Session', 'AppRet', 'X_used', 'control']).mean().reset_index()
parts = [] 
for gx, gby in mean_lineplot_trial_df.groupby(['Animal','Region', 'Session', 'AppRet', 'control']):
    parts.append(list(gx + (spearmanr(gby[['Y_mean','X_used']])[0],)))
Spearman = pd.DataFrame(parts, columns = ['Animal','Region', 'Session', 'AppRet', 'control', 'SCor'])

In [ ]:
%%R -i Spearman -o emm

library(lme4)
library(emmeans)

test <- lmer(SCor ~ AppRet*Region*Session*control+(1|Animal), data=Spearman)

emm <- emmeans(test, ~ Region*control|Session*AppRet)
print(data.frame(pairs(emm, adjust = "bonferroni")))

emm <- emmeans(test, ~ Region*control|Session*AppRet)
#print(emm)
print(data.frame(test(emm, adjust = "bonferroni")))
emm <- as.data.frame(emm)

In [ ]:
def plot_raw_shuffle(
    emm, spearman,
    session="Monster",
    value_col="SCor",
    appret_order=("Approach","Retreat"),
    region_order=("TS","VS"),
    control_order=("original","shuffle"),
    control_labels=("Raw","Shuffle"),
    se_mult=1.0,
    dodge=0.36,
    jitter=0.07,
    connect_vs_ts=True,
    figsize=(4.2, 4.6),
    title=None,
    seed=0
):
    rng = np.random.default_rng(seed)
    colors = {"TS": "0.55", "VS": "#40BFEF"}

    x_base = {c: i for i, c in enumerate(control_order)}
    ctrl_label = dict(zip(control_order, control_labels))
    half = dodge / 2
    roff = {region_order[0]: -half, region_order[1]: +half}

    sp_all = spearman[spearman["Session"] == session]
    em_all = emm[emm["Session"] == session]

    fig, axes = plt.subplots(len(appret_order), 1, sharey=True, figsize=figsize)
    axes = np.atleast_1d(axes)

    for r, appret in enumerate(appret_order):
        ax = axes[r]
        sp = sp_all[sp_all["AppRet"] == appret]
        em = em_all[em_all["AppRet"] == appret]

        ax.grid(False)
        ax.axhline(0, lw=0.8, color="0.85", zorder=0)

        jitter_map = {}
        for ctrl in control_order:
            animals = sp.loc[sp["control"] == ctrl, "Animal"].unique()
            jitter_map[ctrl] = dict(zip(animals, rng.uniform(-jitter, jitter, size=len(animals))))

        if connect_vs_ts:
            for ctrl in control_order:
                g = sp[sp["control"] == ctrl]
                piv = (g.groupby(["Animal","Region"])[value_col].mean()
                         .unstack("Region"))
                for animal, row in piv.dropna(subset=list(region_order)).iterrows():
                    j = jitter_map[ctrl].get(animal, 0.0)
                    x_ts = x_base[ctrl] + roff[region_order[0]] + j
                    x_vs = x_base[ctrl] + roff[region_order[1]] + j
                    ax.plot([x_ts, x_vs], [row[region_order[0]], row[region_order[1]]],
                            lw=1.0, alpha=0.25, color="0.35", zorder=1)

        # points
        for reg in region_order:
            g = sp[sp["Region"] == reg]
            for ctrl in control_order:
                gg = g[g["control"] == ctrl]
                if gg.empty:
                    continue
                xs = np.array([
                    x_base[ctrl] + roff[reg] + jitter_map[ctrl].get(a, 0.0)
                    for a in gg["Animal"].to_numpy()
                ])
                ax.scatter(xs, gg[value_col].to_numpy(),
                           s=18, alpha=0.95, color=colors.get(reg, "0.5"),
                           edgecolors="none", zorder=2)

        # emmean +/- SE
        for reg in region_order:
            g = em[em["Region"] == reg]
            for ctrl in control_order:
                row = g[g["control"] == ctrl].iloc[0]
                x = x_base[ctrl] + roff[reg]
                ax.errorbar(x, row["emmean"], yerr=se_mult * row["SE"],
                            fmt="o", mfc="white", mec="black", mew=1.0,
                            ecolor="black", elinewidth=1.3, capsize=4,
                            markersize=5.5, zorder=5)

        ax.set_xlim(-0.6, len(control_order) - 0.4)
        ax.set_xticks([x_base[k] for k in control_order])
        ax.set_xticklabels([ctrl_label[k] for k in control_order])

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(1.0)
        ax.spines["bottom"].set_linewidth(1.0)
        ax.tick_params(width=1.0, length=4)

        ax.set_title(f"{session} — {appret}")
        ax.set_ylabel(value_col)

    if title:
        fig.suptitle(title, y=0.98)

    fig.tight_layout()
    return fig, axes
fig, axes = plot_raw_shuffle(emm, Spearman, session='Monster', value_col="SCor")
fig.savefig(r"D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\MonsterSpearmanCorr.svg",bbox_inches="tight")

Figure 6K, 6L

In [ ]:
def count_increasing_bins_from_lineplot(
    df: pd.DataFrame,
    x_col: str = "X_used",
    y_col: str = "Y_mean",
    x_max: float = 40.0,
    thresh: float = 0.02,
    goal_align: bool = True,
):
    required = {"Region","Session","AppRet","control","Animal", x_col, y_col}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Missing columns: {missing}")

    group_cols = ["Region","Session","AppRet","control","Animal"]
    if "Trial" in df.columns:
        group_cols.append("Trial")

    d = df.dropna(subset=[x_col, y_col]).copy()
    d[x_col] = pd.to_numeric(d[x_col], errors="coerce")
    d[y_col] = pd.to_numeric(d[y_col], errors="coerce")
    d = d.dropna(subset=[x_col, y_col])

    if goal_align:
        # Retreat flipped
        is_ret = d["AppRet"].astype(str).str.lower().str.startswith("ret")
        d["X_goal"] = np.where(is_ret, x_max - d[x_col].to_numpy(dtype=float), d[x_col].to_numpy(dtype=float))
        x_use = "X_goal"
    else:
        d["X_goal"] = d[x_col]
        x_use = x_col

    rows = []
    for keys, g in d.groupby(group_cols, sort=False):
        g2 = (g.groupby(x_use, as_index=False, observed=True)[y_col]
                .mean()
                .sort_values(x_use))

        x = g2[x_use].to_numpy(dtype=float)
        y = g2[y_col].to_numpy(dtype=float)

        if len(x) < 2:
            rows.append({**dict(zip(group_cols, keys)),
                         "n_bins": int(len(x)),
                         "n_total_steps": 0,
                         "n_increasing": 0,
                         "frac_increasing": np.nan,
                         "thresh": float(thresh)})
            continue

        dx = np.diff(x)
        dy = np.diff(y)
        ok = dx > 0
        slopes = np.full_like(dy, np.nan, dtype=float)
        slopes[ok] = dy[ok] / dx[ok]

        inc = slopes > thresh
        n_total_steps = int(np.sum(~np.isnan(slopes)))
        n_increasing  = int(np.nansum(inc))

        rows.append({**dict(zip(group_cols, keys)),
                     "n_bins": int(len(x)),
                     "n_total_steps": n_total_steps,
                     "n_increasing": n_increasing,
                     "frac_increasing": (n_increasing / n_total_steps) if n_total_steps else np.nan,
                     "thresh": float(thresh)})

    return pd.DataFrame(rows)
inc_trial  = count_increasing_bins_from_lineplot(lineplot_trial_df, thresh=0.02, x_max=40)

In [ ]:
%%R -i inc_trial -o emm

library(lme4)
library(emmeans)

test <- lmer(n_increasing ~ AppRet*Region*Session*control+(1|Animal), data=inc_trial)

emm <- emmeans(test, ~ Region*control|Session*AppRet)
print(data.frame(pairs(emm, adjust = "bonferroni")))

emm <- emmeans(test, ~ Region*control|Session*AppRet)
#print(emm)
print(data.frame(test(emm, adjust = "bonferroni")))
emm <- as.data.frame(emm)

In [ ]:
value_col = "n_increasing" 

sp_plot = (
    inc_trial
      .groupby(["Session","AppRet","Region","control","Animal"], observed=True, as_index=False)
      .agg(**{value_col: (value_col, "mean")})
)

emm_plot = (
    sp_plot
      .groupby(["Session","AppRet","Region","control"], observed=True)[value_col]
      .agg(emmean="mean",
           SE=lambda s: s.std(ddof=1) / np.sqrt(s.count()))
      .reset_index()
)

fig, axes = plot_raw_shuffle(
    emm=emm,
    spearman=sp_plot,
    session="Monster",
    value_col=value_col
)
fig.savefig(r"D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\BinsIncreasingOver.svg",bbox_inches="tight")

FIgure 6I, 6J

In [ ]:
X_col     = "X"
Y_col     = "romb_cum_Z_zscore"
TRIAL_COL = "Trial"
TIME_COL  = "Time to Shelter"
x_min, x_max = 30, 40

nbins = 40
n_perm = 1000
shuffle_seed = 123

groups = [
    (key, sdf)
    for key, sdf in df.groupby(["Region", "Session", "AppRet"], sort=False)
    if not sdf.empty
]

trial_rows_raw = []
trial_rows_shuf = []

rng0 = np.random.RandomState(shuffle_seed)

for (site, session, appret), sub_df in groups:

    sub = sub_df.dropna(subset=[X_col, Y_col, TIME_COL, TRIAL_COL, "Animal"]).copy()
    sub[X_col] = pd.to_numeric(sub[X_col], errors="coerce")
    sub = sub.dropna(subset=[X_col, Y_col, TIME_COL])

    sub = sub[(sub[X_col] >= x_min) & (sub[X_col] <= x_max)]
    if sub.empty:
        continue

    if TRIAL_COL not in sub.columns:
        raise KeyError(f"Expected trial column '{TRIAL_COL}' in dataframe")

    sub = sub.sort_values(["Animal", TRIAL_COL, TIME_COL])

    if not np.issubdtype(sub[X_col].dtype, np.integer):
        sub[X_col] = sub[X_col].clip(lower=x_min, upper=np.nextafter(x_max, -np.inf))
        bins = np.linspace(x_min, x_max, nbins + 1)
        sub["X_bin"] = pd.cut(sub[X_col], bins=bins,
                              labels=False, include_lowest=True, right=True)
        centers = (bins[:-1] + bins[1:]) / 2
        center_map = pd.Series(centers, index=pd.RangeIndex(nbins))
        sub["X_mid"] = sub["X_bin"].map(center_map)
        label_col = "X_mid"
    else:
        label_col = X_col

    raw_trial = (
        sub.groupby(["Animal", TRIAL_COL, label_col], observed=True)[Y_col]
           .mean()
           .reset_index()
           .rename(columns={label_col: "X_used", Y_col: "Y_mean"})
    )
    raw_trial["Region"]  = site
    raw_trial["Session"] = session
    raw_trial["AppRet"]  = appret
    raw_trial["control"] = "original"
    trial_rows_raw.append(raw_trial)
    base = sub[["Animal", TRIAL_COL, TIME_COL, label_col, Y_col]].copy()

    acc_sum = None
    acc_n   = None

    for _ in range(n_perm):
        rng = np.random.RandomState(rng0.randint(0, 2**31 - 1))

        tmp = base.copy()
        tmp["Y_shuf"] = np.nan

        # circular shift Y
        for (ani, trial), g in tmp.groupby(["Animal", TRIAL_COL], sort=False):
            vals = g[Y_col].to_numpy()
            n_rows = len(vals)
            if n_rows > 1:
                shift = rng.randint(0, n_rows)
                vals = np.roll(vals, shift)
            tmp.loc[g.index, "Y_shuf"] = vals

        m = (
            tmp.groupby(["Animal", TRIAL_COL, label_col], observed=True)["Y_shuf"]
               .mean()
               .reset_index()
               .rename(columns={label_col: "X_used", "Y_shuf": "Y_perm"})
        )

        s = m.set_index(["Animal", TRIAL_COL, "X_used"])["Y_perm"]

        if acc_sum is None:
            acc_sum = s.copy()
            acc_n   = pd.Series(1.0, index=s.index)
        else:
            acc_sum = acc_sum.add(s, fill_value=0.0)
            acc_n   = acc_n.add(pd.Series(1.0, index=s.index), fill_value=0.0)

    shuf_mean = (acc_sum / acc_n).rename("Y_mean").reset_index()
    shuf_mean["Region"]  = site
    shuf_mean["Session"] = session
    shuf_mean["AppRet"]  = appret
    shuf_mean["control"] = "shuffle"
    shuf_mean["n_perm_eff"] = acc_n.values

    trial_rows_shuf.append(shuf_mean)

raw_trial_df  = pd.concat(trial_rows_raw,  ignore_index=True) if trial_rows_raw  else pd.DataFrame()
shuf_trial_df = pd.concat(trial_rows_shuf, ignore_index=True) if trial_rows_shuf else pd.DataFrame()

lineplot_trial_df = pd.concat([raw_trial_df, shuf_trial_df], ignore_index=True)

lineplot_animal_df = (
    lineplot_trial_df
      .groupby(["Region","Session","AppRet","control","Animal","X_used"], observed=True, as_index=False)
      .agg(Y_mean=("Y_mean","mean"))
)


mean_lineplot_trial_df = lineplot_trial_df.groupby(['Animal','Region', 'Session', 'AppRet', 'X_used', 'control']).mean().reset_index()
parts = [] 
for gx, gby in mean_lineplot_trial_df.groupby(['Animal','Region', 'Session', 'AppRet', 'control']):
    parts.append(list(gx + (spearmanr(gby[['Y_mean','X_used']])[0],)))
Spearman = pd.DataFrame(parts, columns = ['Animal','Region', 'Session', 'AppRet', 'control', 'SCor'])

In [ ]:
%%R -i Spearman -o emm

library(lme4)
library(emmeans)

test <- lmer(SCor ~ AppRet*Region*Session*control+(1|Animal), data=Spearman)

emm <- emmeans(test, ~ Region*control|Session*AppRet)
print(data.frame(pairs(emm, adjust = "bonferroni")))

emm <- emmeans(test, ~ Region*control|Session*AppRet)
#print(emm)
print(data.frame(test(emm, adjust = "bonferroni")))
emm <- as.data.frame(emm)

In [ ]:
fig, axes = plot_raw_shuffle(emm, Spearman, value_col="SCor")
fig.savefig(r"D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\Cm10SpearmanCorr.svg",bbox_inches="tight")